# 04. Uji Validasi Silang / "5-Fold Walk-Forward CV (Parameter Tetap / Harsh Environment)".

**Naskah Akademik & Pembuktian Teoretis Skripsi**  
*Program Studi Sistem Informasi / Ilmu Komputer*

Notebook ini didedikasikan untuk melakukan evaluasi ketahanan ekstrim (*Harsh Environment Testing*) pada model Champion yang telah diekstraksi di Notebook 03. Pengujian dilakukan melalui teknik **5-Fold Walk-Forward Cross Validation** dengan membandingkannya secara *head-to-head* melawan model **Seasonal Naive**. 

## Tujuan Evaluasi
1. Menguji stabilitas performa lintas waktu (mencegah overfitting in-sample).
2. Mengeksekusi pengujian dengan jendela statis 8-minggu pada setiap fold.
3. Mengukur efikasi model Champion menggunakan metrik baku: **WMAPE**, **MAPE**, **RMSE**, dan **MAE**.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import warnings
import json
import os

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
os.makedirs('Visualisasi_Gambar', exist_ok=True)

# Load Data
df = pd.read_csv('../01_data_ITA-Cake-Shop/Data_Output/weekly_clean_final.csv', parse_dates=['date'], index_col='date')
df = df.resample('W-MON').sum()
df = df[df.index <= '2026-02-09'] # Exclude truncation

print(f"Data loaded: {df.shape[0]} weeks from {df.index.min().date()} to {df.index.max().date()}")


Data loaded: 155 weeks from 2023-02-20 to 2026-02-02


## 1. Definisi Parameter Champion Statis
Mengimpor parameter statis yang terpilih dari hasil komputasi Grid Search Notebook 03. Parameter ini dikunci dan tidak dioptimasi ulang (`optimized=False`) pada saat *cross validation* untuk mencerminkan kondisi produksi nyata.

In [2]:
# Membaca parameter champion secara dinamis
json_path = 'Data_Output/champion_parameters.json'
with open(json_path, 'r') as f:
    champion_params_raw = json.load(f)

champion_params = {}
for cat, p in champion_params_raw.items():
    champion_params[cat] = {
        'model': p.get('seasonal', 'multiplicative'),
        'm': p.get('m', 26),
        'trend': p.get('trend', None),
        'alpha': p.get('alpha', 0.0),
        'beta': None if p.get('trend') is None else p.get('beta', 0.0),
        'gamma': p.get('gamma', 0.0),
        'init': p.get('init', 'estimated') # Mengambil inisialisasi persis dari JSON
    }

# Fungsi metrik evaluasi
def get_metrics(y_true, y_pred, name):
    mask = y_true > 0
    if mask.sum() == 0:
        return {'Model': name, 'WMAPE': 0, 'MAPE': 0, 'RMSE': 0, 'MAE': 0}
    wmape = np.sum(np.abs(y_true - y_pred)) / np.sum(y_true) * 100
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    rmse = np.sqrt(np.mean((y_true - y_pred)**2))
    mae = np.mean(np.abs(y_true - y_pred))
    return {'Model': name, 'WMAPE': wmape, 'MAPE': mape, 'RMSE': rmse, 'MAE': mae}

## 2. 5-Fold Walk-Forward Cross Validation
Menerapkan pengujian *walk-forward* dengan jendela geser (*expanding window*) dan ukuran *holdout* pengujian statis 8 minggu (sekitar 2 bulan) per *fold*.

In [3]:
results = []
n_splits = 5
test_size = 8
total_len = len(df)

# Visualisasi Skema Walk-Forward Cross Validation Timeline
fig, ax = plt.subplots(figsize=(13, 5))
y_positions = list(range(n_splits, 0, -1))
for idx in range(n_splits):
    fe = total_len - (n_splits - 1 - idx) * test_size
    tr_end = fe - test_size
    y = y_positions[idx]
    ax.barh(y, tr_end, left=0, height=0.45, color='#2b6cb0', alpha=0.9,
            edgecolor='#1a365d', label='Data Latih (In-Sample Training)' if idx == 0 else "")
    ax.barh(y, test_size, left=tr_end, height=0.45, color='#e53e3e', alpha=0.9,
            edgecolor='#9b2c2c', label='Data Uji (Out-of-Sample Test Window: 8 Minggu)' if idx == 0 else "")
    ax.text(tr_end / 2, y, f'{tr_end} Minggu Latih', ha='center', va='center', color='white', fontweight='bold', fontsize=9)
    ax.text(tr_end + test_size / 2, y, f'Fold {idx+1}\n(8 W)', ha='center', va='center', color='white', fontweight='bold', fontsize=8)
ax.set_yticks(y_positions)
ax.set_yticklabels([f'Fold {i}' for i in range(1, n_splits + 1)], fontweight='bold')
ax.set_xlabel('Indeks Waktu Minggu (0 s/d 155)')
ax.set_xlim(-2, total_len + 5)
ax.set_title('Skema 5-Fold Walk-Forward Cross Validation (Expanding Window Protocol)', fontsize=13, fontweight='bold', pad=15)
ax.legend(loc='upper left', frameon=True)
plt.tight_layout()
plt.savefig('Visualisasi_Gambar/cv_walk_forward_scheme.png', dpi=300, bbox_inches='tight')
plt.show()

# Eksekusi Cross Validation & Visualisasi Prediksi Per Kategori
for cat in ['Cake', 'Japanese', 'Mille Crepes']:
    series = df[cat].copy()
    params = champion_params[cat]
    fold_results = []
    slug = cat.lower().replace(' ', '')
    
    fig, axes = plt.subplots(n_splits, 1, figsize=(14, 15), sharey=True)
    
    for i in range(n_splits):
        end_idx = total_len - (n_splits - i) * test_size
        train = series.iloc[:end_idx]
        test = series.iloc[end_idx : end_idx + test_size]
        
        # Seasonal Naive
        m_baseline = 26  # Benchmark kalender semesteran standar (26 pekan)
        if len(train) >= m_baseline:
            y_snaive = train.iloc[-m_baseline:].values
            if len(y_snaive) < test_size:
                y_snaive = np.tile(y_snaive, int(np.ceil(test_size / len(y_snaive))))[:test_size]
            else:
                y_snaive = y_snaive[:test_size]
        else:
            y_snaive = np.full(test_size, train.mean())
            
        # Holt-Winters Champion
        train_hw = train.copy()
        if params['model'] == 'mul':
            train_hw = train_hw.replace(0, 0.1)
            
        hw = ExponentialSmoothing(train_hw, 
                                  trend=params['trend'], 
                                  seasonal=params['model'], 
                                  seasonal_periods=params['m'],
                                  initialization_method=params['init']).fit(
                                      smoothing_level=params['alpha'],
                                      smoothing_trend=params['beta'],
                                      smoothing_seasonal=params['gamma'],
                                      optimized=False
                                  )
        
        y_hw = hw.forecast(test_size)
        
        # Metrics
        res_snaive = get_metrics(test.values, y_snaive, 'Seasonal Naive')
        res_hw = get_metrics(test.values, y_hw.values, 'Champion HW')
        
        for r in [res_snaive, res_hw]:
            r['Category'] = cat
            r['Fold'] = i + 1
            fold_results.append(r)
            
        # Plot Fold
        ax = axes[i]
        ctx_train = train.iloc[-12:]
        ax.plot(ctx_train.index, ctx_train.values, color='#718096', lw=1.5, marker='o', ms=3, label='Histori Latih (12 Minggu Terakhir)')
        ax.plot(test.index, test.values, color='#1a202c', lw=2.2, marker='s', ms=4, label='Aktual Uji (Ground Truth)')
        cat_color = {'Cake': '#2b6cb0', 'Japanese': '#d69e2e', 'Mille Crepes': '#319795'}.get(cat, '#2b6cb0')
        ax.plot(test.index, y_hw.values, color=cat_color, lw=2.0, marker='^', ms=4, label=f'HW Multiplicative (m={params["m"]}) [MAPE: {res_hw["MAPE"]:.1f}%]')
        ax.plot(test.index, y_snaive, color='#e53e3e', lw=1.8, marker='x', ms=4, linestyle='--', label=f'Seasonal Naive Baseline [MAPE: {res_snaive["MAPE"]:.1f}%]')
        ax.set_title(f'{cat} — Fold {i+1} ({test.index[0].strftime("%d %b %Y")} s/d {test.index[-1].strftime("%d %b %Y")}) | HW RMSE: {res_hw["RMSE"]:.2f} vs SN RMSE: {res_snaive["RMSE"]:.2f}',
                     fontsize=10, fontweight='bold', loc='left')
        ax.set_ylabel('Qty (Pcs)')
        ax.legend(loc='upper right', frameon=True, fontsize=8)

    axes[-1].set_xlabel('Tanggal Evaluasi Mingguan (W-MON)')
    plt.tight_layout()
    plt.savefig(f'Visualisasi_Gambar/cv_fold_predictions_{slug}.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    results.extend(fold_results)

df_cv = pd.DataFrame(results)
display(df_cv.head())

,Model,WMAPE,MAPE,RMSE,MAE,Category,Fold
0,Seasonal Naive,36.467236,43.301131,54.032398,48.000000,Cake,1
1,Champion HW,26.629278,28.811779,42.871099,35.050787,Cake,1
2,Seasonal Naive,11.826087,11.242014,20.068632,17.000000,Cake,2
3,Champion HW,33.009311,34.432231,49.807737,47.450885,Cake,2
4,Seasonal Naive,25.563525,25.248283,32.272037,31.187500,Cake,3


## 3. Komparasi Head-to-Head & Kesimpulan Akademik
Berdasarkan tabel evaluasi rata-rata 5-Fold, Champion Holt-Winters terbukti secara konsisten mengalahkan *Seasonal Naive* pada ketiga metrik error (WMAPE, RMSE, MAE). Model terhindar dari *underfitting* (karena tidak memaksakan tren aditif yang berbalik arah) dan juga terhindar dari *overfitting* (melalui penguncian $\gamma=0.0000$). Parameter statis ini siap diangkat ke arsitektur operasional (*Notebook 05*).

In [4]:
df_cv.to_csv('Data_Output/cv_fold_metrics_detailed.csv', index=False)
print('Saved cv_fold_metrics_detailed.csv')
# Rekapitulasi Leaderboard
df_leaderboard = df_cv.groupby(['Category', 'Model'])[['WMAPE', 'MAPE', 'RMSE', 'MAE']].mean().reset_index()
df_leaderboard.to_csv('Data_Output/leaderboard_cv_final.csv', index=False)
print('Saved leaderboard_cv_final.csv')
display(df_leaderboard)

# Visualisasi Komparasi Multidimensi (cv_metric_comparison.png)
categories_plot = ['Cake', 'Japanese', 'Mille Crepes']
hw_means = df_cv[df_cv['Model'] == 'Champion HW'].groupby('Category')[['MAPE', 'WMAPE', 'RMSE', 'MAE']].mean().reindex(categories_plot)
sn_means = df_cv[df_cv['Model'] == 'Seasonal Naive'].groupby('Category')[['MAPE', 'WMAPE', 'RMSE', 'MAE']].mean().reindex(categories_plot)

fig, axes = plt.subplots(2, 2, figsize=(14, 10.5))
x = np.arange(len(categories_plot))
width = 0.35

panels = [
    (axes[0, 0], 'MAPE', '(A) Perbandingan Masked MAPE (%)', 'MAPE (%)', '#2b6cb0', '#1a365d', '{:.1f}%'),
    (axes[0, 1], 'WMAPE', '(B) Perbandingan Volume-Weighted MAPE (WMAPE %)', 'WMAPE (%)', '#319795', '#234e52', '{:.1f}%'),
    (axes[1, 0], 'RMSE', '(C) Perbandingan Root Mean Squared Error (RMSE)', 'RMSE (Pcs)', '#2b6cb0', '#1a365d', '{:.2f}'),
    (axes[1, 1], 'MAE', '(D) Perbandingan Mean Absolute Error (MAE)', 'MAE (Pcs)', '#319795', '#234e52', '{:.2f}')
]

for ax, col_m, title_m, ylabel_m, c_hw, ec_hw, fmt_str in panels:
    r1 = ax.bar(x - width/2, hw_means[col_m], width, label='Holt-Winters Champion', color=c_hw, edgecolor=ec_hw)
    r2 = ax.bar(x + width/2, sn_means[col_m], width, label='Seasonal Naive Baseline (m=26)', color='#e53e3e', edgecolor='#9b2c2c')
    ax.set_title(title_m, fontweight='bold', fontsize=11)
    ax.set_xticks(x)
    ax.set_xticklabels(categories_plot, fontweight='bold')
    ax.set_ylabel(ylabel_m)
    max_h = max(hw_means[col_m].max(), sn_means[col_m].max())
    ax.set_ylim(0, max_h * 1.25)
    ax.legend(loc='upper left', frameon=True, fontsize=8.5)
    for r in list(r1) + list(r2):
        ax.annotate(fmt_str.format(r.get_height()), xy=(r.get_x() + r.get_width()/2, r.get_height()),
                    xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=8.5, fontweight='bold')

fig.suptitle('Evaluasi Komparasi Kinerja Holt-Winters Champion vs Seasonal Naive Baseline (5-Fold CV Average)', 
             fontsize=13.5, fontweight='bold', y=0.99)
plt.tight_layout()
plt.savefig('Visualisasi_Gambar/cv_metric_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

Saved cv_fold_metrics_detailed.csv
Saved leaderboard_cv_final.csv
    Category          Model     WMAPE      MAPE      RMSE       MAE
        Cake    Champion HW 21.446461 22.111797 33.071641 28.461438
        Cake Seasonal Naive 24.422806 25.836320 35.635464 31.387500
    Japanese    Champion HW 24.727866 26.038643 20.811915 18.317047
    Japanese Seasonal Naive 45.101444 47.527601 38.066362 34.059375
Mille Crepes    Champion HW 22.806384 25.081317 17.243111 13.055483
Mille Crepes Seasonal Naive 34.103020 36.136093 22.003330 19.925000
